Tarea 1: Cliente de Claude API (infrastructure/llm/anthropic.py)
Implementa la clase que cumple el Protocol LLMProvider de domain/interfaces.py. Debe poder recibir una lista de Message y devolver una respuesta. Usa el SDK de Anthropic (anthropic package). Usa settings.default_model para el modelo y settings.anthropic_api_key para la key. Escribe un test unitario con el MockLLMProvider que ya tienes en conftest, y un test de integración que haga una llamada real a la API (con @pytest.mark.integration).

Tarea 2: Cliente de arXiv API (infrastructure/data/arxiv.py)
arXiv tiene una API REST gratuita que devuelve XML. Implementa una función que reciba un query string (ej. "LLM agents") y un número máximo de resultados, haga la request con httpx, parsee el XML, y devuelva una lista de Paper (tu modelo de dominio). El endpoint es http://export.arxiv.org/api/query. Escribe un test de integración que busque 3 papers sobre "LLM agents" y verifique que devuelve objetos Paper válidos.

Tarea 3: Descarga y extracción de PDFs (application/services/ingestion_service.py)
Crea la función de ingesta que toma un Paper, descarga su PDF usando paper.pdf_url, extrae el texto con PyMuPDF (fitz), y devuelve el texto crudo. No hagas chunking todavía — eso es tarea de semana 2. Solo descarga + extracción de texto. Guarda los PDFs en data/papers/ (crea la carpeta si no existe, agrégala a .gitignore).

In [1]:
# Use this initial code to work in the notebook as if it were a module, that
# is, to be able to export classes and functions from other subpackages.

import os
import sys

package_path = os.path.abspath(".").split(os.sep + "notebooks")[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

# Task 1

In [ ]:
from collections.abc import AsyncIterator
from typing import Protocol


# Protocolo estableciod en domain/interfaces
class LLMProvider(Protocol):
    """Contract for any LLM provider (Claude, Gemini, etc.)."""

    async def generate(self, messages: list[Message]) -> str:
        """Generate a response from a list of messages."""
        ...

    async def stream(self, messages: list[Message]) -> AsyncIterator[str]:
        """Stream a response token by token."""
        ...


from anthropic import AsyncAnthropic
from src.researchos.domain import Message

from researchos.config import settings
from researchos.domain.exceptions import GenerationError


class AnthropicLLM:
    """Contract for Claude provider."""

    def __init__(self):
        self.client = AsyncAnthropic(api_key=settings.anthropic_api_key)
        self.model_id = settings.default_model
        self.temperature = settings.temperature
        self.max_tokens = settings.max_tokens

    def _format_messages(self, messages: list[Message]) -> tuple[str | None, list[dict]]:
        """
        Separa el system prompt (si existe) y formatea los mensajes
        para el esquema que espera Anthropic.
        """
        system_prompt = None
        formatted = []

        for msg in messages:
            if msg.role == "system":
                system_prompt = msg.content
            else:
                formatted.append({"role": msg.role, "content": msg.content})

        return system_prompt, formatted

    async def generate(self, messages: list[Message]) -> str:
        """Generate a response from a list of messages."""

        system, formatted_msgs = self._format_messages(messages)

        response = await self.client.messages.create(
            model=self.model_id,
            max_tokens=self.max_tokens,
            system=system or "",
            messages=formatted_msgs,
            temperature=self.temperature,
        )

        if response.content and len(response.content) > 0:
            return response.content[0].text
        else:
            raise GenerationError("Claude returned empty response")

    async def stream(self, messages: list[Message]) -> AsyncIterator[str]:
        """Stream a response token by token."""

        system, formatted_msgs = self._format_messages(messages)

        async with self.client.messages.stream(
            model=self.model_id,
            max_tokens=self.max_tokens,
            system=system or "",
            messages=formatted_msgs,
            temperature=self.temperature,
        ) as stream:
            async for text in stream.text_stream:
                yield text

# Task 2